# Autoformer / PatchTST 消融实验（重做版）

## 实验目的

消融实验不是重新比较所有模型，而是从当前表现较强、结构特点最明确的 **Autoformer** 和 **PatchTST** 出发，有控制地移除或替换某个关键部件，观察预测误差如何变化。这样可以回答：模型取得较好结果究竟依赖哪些设计，而不是只报告“哪个模型更好”。

本 notebook 将两个原始基线和四个消融变体放入同一个 run tag，在相同数据、随机种子、训练轮数、早停规则和优化器参数下重新训练，避免拿不同实验批次的结果直接比较。

## 本次包含的 6 个模型

| 模型标识 | 类型 | 保留的主要结构 | 被改变的部件 |
| --- | --- | --- | --- |
| `autoformer_ablation_base` | Autoformer 基线 | 序列分解、Auto-Correlation、趋势/季节分支 | 无 |
| `autoformer_no_decomp` | Autoformer 消融 | Auto-Correlation 和其余网络结构 | 移除序列分解，趋势分支置零 |
| `autoformer_no_autocorr` | Autoformer 消融 | 序列分解和其余网络结构 | 用标准多头自注意力替换 Auto-Correlation |
| `patchtst_ablation_base` | PatchTST 基线 | Patching、Channel Independence、Transformer 编码器 | 无 |
| `patchtst_no_patch` | PatchTST 消融 | Channel Independence 和 Transformer 编码器 | 取消 patch 切分，改为逐时间点投影 |
| `patchtst_channel_mix` | PatchTST 消融 | Patching 和 Transformer 编码器 | 取消 Channel Independence，联合混合全部变量 |

## 被检验的四个关键部件

1. **Series Decomposition（序列分解）**：Autoformer 使用移动平均将输入拆分为趋势项和季节项。`autoformer_no_decomp` 用于判断显式分解是否能降低长期预测难度。
2. **Auto-Correlation（自相关机制）**：通过 FFT 和主要时延捕捉周期依赖。`autoformer_no_autocorr` 将其替换为标准多头注意力，用于判断周期相关建模是否优于普通注意力。
3. **Patching（片段化）**：PatchTST 把连续时间点组成局部片段，以较少 token 表达局部变化。`patchtst_no_patch` 用于检验 patch 表示的贡献。
4. **Channel Independence（通道独立）**：PatchTST 对每个变量独立编码并共享模型参数，减少变量间分布差异造成的干扰。`patchtst_channel_mix` 保留 patching，只改为多变量联合编码，因此主要检验通道独立策略。

## 实验范围

- 数据集：ETTh1、ETTm1
- 回看窗口：96
- 预测步长：96、336，分别代表中期和长期预测
- 模型数量：2 个基线 + 4 个消融变体
- 正式矩阵：`2 数据集 × 2 步长 × 6 模型 = 24 组`
- 随机种子：42

> 默认不会启动训练。先检查参数表、任务矩阵和模型前向形状；确认无误后，将 `RUN_EXPERIMENTS=True`。每组结果独立保存，中断后重新运行会跳过已经完成的实验。

## 0. 配置、调优参数与安全开关

### 参数继承原则

消融实验应当只改变目标部件，其余参数保持不变。本实验继承 `docs/best_model_params.md` 中基于 **ETTh1 h96 验证集 best_val_loss** 选出的 Autoformer 和 PatchTST 最优参数。基线与对应变体共享同一套结构参数，避免把超参数差异误判成模块贡献。

### Autoformer 统一结构参数

| 参数 | 数值 | 含义 |
| --- | ---: | --- |
| `d_model` | 64 | 隐表示维度 |
| `n_heads` | 4 | 注意力/自相关头数 |
| `n_encoder_layers` | 2 | 编码器层数 |
| `n_decoder_layers` | 1 | 解码器层数 |
| `d_ff` | 128 | 前馈网络维度 |
| `factor` | 3 | Auto-Correlation 主要时延数量控制参数 |
| `kernel_size` | 25 | 移动平均分解窗口 |
| `dropout` | 0.1 | Dropout 比例 |

### PatchTST 统一结构参数

| 参数 | 数值 | 含义 |
| --- | ---: | --- |
| `d_model` | 64 | 每个 patch 的隐表示维度 |
| `n_heads` | 8 | 多头注意力头数 |
| `n_layers` | 2 | Transformer 编码层数 |
| `d_ff` | 128 | 前馈网络维度 |
| `patch_len` | 32 | 每个时间片段包含的时间点数 |
| `stride` | 8 | 相邻 patch 的滑动步长 |
| `dropout` | 0.1 | Dropout 比例 |

### 统一训练参数

| 参数 | 数值 |
| --- | ---: |
| 最大训练轮数 | 50 |
| Early Stopping patience | 10 |
| Batch size | 128 |
| Learning rate | 0.001 |
| Weight decay | 0.00001 |
| 全量样本 | `sample_limit=0` |

下面的代码会显示实际使用的参数，并在训练前检查基线与变体是否完整继承上述配置。若参数发生漂移，Notebook 会直接报错。

In [ ]:
from pathlib import Path
from types import SimpleNamespace
from collections import defaultdict
import json
import sys
import time

ROOT = Path.cwd().resolve()
for _ in range(8):
    if (ROOT / 'scripts').is_dir() and (ROOT / 'models').is_dir():
        break
    if ROOT.parent == ROOT:
        raise FileNotFoundError('无法定位项目根目录：未找到 scripts/ 和 models/。')
    ROOT = ROOT.parent
else:
    raise FileNotFoundError('无法定位项目根目录：未找到 scripts/ 和 models/。')

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f'Project root: {ROOT}')

CONFIG_PATH = ROOT / 'configs' / 'ablation_rerun_etth1_ettm1.json'
config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))

# 正式运行：USE_SMOKE_CONFIG=False，并手动改为 RUN_EXPERIMENTS=True。
USE_SMOKE_CONFIG = True
RUN_EXPERIMENTS = True
RUN_ONLY_FIRST_N = None
MISSING_DATA_POLICY = 'skip'  # 'raise' 或 'skip'

SMOKE_TRAINING_CONFIG = {
    'epochs': 1,
    'patience': 1,
    'batch_size': 16,
    'lr': 0.001,
    'weight_decay': 0.00001,
}

if USE_SMOKE_CONFIG:
    config.update({
        'datasets': 'ETTh1',
        'horizons': '96',
        **SMOKE_TRAINING_CONFIG,
        'sample_limit': 64,
        'run_tag': 'ablation_rerun_smoke',
        'no_tensorboard': True,
    })

print(f"Mode: {'SMOKE' if USE_SMOKE_CONFIG else 'FORMAL'}")
print(f"RUN_EXPERIMENTS={RUN_EXPERIMENTS}")
config

## 1. 生成实验计划

任务计划按“数据集 → 预测步长 → 模型”展开。正式情况下应得到 24 行：每个数据集和预测步长组合都包含一个 Autoformer 基线、两个 Autoformer 消融、一个 PatchTST 基线和两个 PatchTST 消融。

状态含义：

- `pending`：处理后数据存在，但该实验尚未产生完整结果。
- `completed`：结果数组与 summary JSON 均已存在，续跑时自动跳过。
- `missing_data`：对应数据集或预测步长缺少 `train/val/test.npz`，正式训练前必须处理。

这里先检查任务数量与数据完整性，不会启动训练。

In [ ]:
from scripts.run_experiments import (
    ABLATION_MODEL_CONFIGS,
    ABLATION_TUNED_MODEL_CONFIGS,
    ABLATION_TUNED_TRAINING_CONFIG,
    MODEL_BUILDERS,
    parse_csv_list,
    parse_int_list,
)
import pandas as pd

datasets = parse_csv_list(config['datasets'])
horizons = parse_int_list(config['horizons'])
models = parse_csv_list(config['models'])
expected_models = {
    'autoformer_ablation_base',
    'autoformer_no_decomp',
    'autoformer_no_autocorr',
    'patchtst_ablation_base',
    'patchtst_no_patch',
    'patchtst_channel_mix',
}
if set(models) != expected_models:
    raise ValueError(f'消融模型集合不完整：{models}')
unknown_models = sorted(set(models) - set(MODEL_BUILDERS))
if unknown_models:
    raise ValueError(f'Unknown models: {unknown_models}')

# 基线及其变体必须共享同一套调优后结构参数。
for family, family_models in {
    'autoformer': ['autoformer_ablation_base', 'autoformer_no_decomp', 'autoformer_no_autocorr'],
    'patchtst': ['patchtst_ablation_base', 'patchtst_no_patch', 'patchtst_channel_mix'],
}.items():
    expected_config = ABLATION_TUNED_MODEL_CONFIGS[family]
    for model_name in family_models:
        if ABLATION_MODEL_CONFIGS[model_name] != expected_config:
            raise ValueError(f'{model_name} 未完整继承 {family} 调优参数。')

actual_training_config = {
    key: config[key] for key in ABLATION_TUNED_TRAINING_CONFIG
}
expected_training_config = (
    SMOKE_TRAINING_CONFIG
    if USE_SMOKE_CONFIG
    else ABLATION_TUNED_TRAINING_CONFIG
)
if actual_training_config != expected_training_config:
    mode_name = 'Smoke' if USE_SMOKE_CONFIG else '正式'
    raise ValueError(
        f'{mode_name}训练参数与预期不一致：{actual_training_config} '
        f'!= {expected_training_config}'
    )

print(f"参数来源: {config.get('parameter_source')}")
print(f"当前模式: {'SMOKE' if USE_SMOKE_CONFIG else 'FORMAL'}")
print(f'当前训练参数: {actual_training_config}')
if USE_SMOKE_CONFIG:
    print('说明：Smoke 模式只缩小训练预算；六个模型仍继承调优后的结构参数。')
display(pd.DataFrame([
    {'model': model_name, **model_config}
    for model_name, model_config in ABLATION_MODEL_CONFIGS.items()
]).set_index('model'))

data_dir = Path(config['data_dir'])
if not data_dir.is_absolute():
    data_dir = ROOT / data_dir
run_tag_dir = config['run_tag'] or 'default'

def has_processed_data(dataset_name, horizon):
    h_dir = data_dir / dataset_name / f'h{horizon}'
    return all((h_dir / split).exists() for split in ('train.npz', 'val.npz', 'test.npz'))

def result_paths(dataset_name, horizon, model_name):
    tag = f"_{config['run_tag']}" if config['run_tag'] else ''
    run_name = f'{dataset_name}_h{horizon}_{model_name}{tag}'
    out_dir = ROOT / 'results' / f'h{horizon}' / dataset_name / model_name / run_tag_dir
    return out_dir / f'{run_name}_results.npy', out_dir / f'{run_name}_summary.json'

def build_plan():
    rows = []
    for dataset_name in datasets:
        for horizon in horizons:
            data_ok = has_processed_data(dataset_name, horizon)
            for model_name in models:
                result_path, summary_path = result_paths(dataset_name, horizon, model_name)
                completed = result_path.exists() and summary_path.exists()
                status = 'completed' if completed else ('pending' if data_ok else 'missing_data')
                rows.append({
                    'dataset': dataset_name,
                    'horizon': horizon,
                    'model': model_name,
                    'status': status,
                    'result': str(result_path.relative_to(ROOT)),
                    'summary': str(summary_path.relative_to(ROOT)),
                })
    return rows

plan = build_plan()
counts = defaultdict(int)
for row in plan:
    counts[row['status']] += 1
print(f'实验矩阵: {len(datasets)} datasets × {len(horizons)} horizons × {len(models)} models = {len(plan)} runs')
print(dict(counts))

try:
    import pandas as pd
    plan_df = pd.DataFrame(plan)
    display(plan_df.groupby(['dataset', 'status']).size().reset_index(name='runs'))
    display(plan_df)
except ImportError:
    for row in plan:
        print(row)

## 2. 模型前向与参数量预检查

正式训练前先做结构级检查，避免长时间训练后才发现输出维度错误。

所有模型接收形状为 `(batch, lookback, variables)` 的输入。本项目两个 ETT 数据集均有 7 个变量，因此测试输入为 `(2, 96, 7)`；模型输出必须为 `(2, horizon, 7)`。

本单元还列出每个模型的可训练参数量。消融后参数量可能变化，这是结构改变的自然结果，但基线与变体除目标部件外应保持相同的隐藏维度、层数、注意力头数和训练预算。

In [ ]:
import torch
from scripts.run_experiments import count_parameters

shape_rows = []
for horizon in horizons:
    x = torch.randn(2, 96, 7)
    for model_name in models:
        model = MODEL_BUILDERS[model_name](7, horizon).eval()
        with torch.no_grad():
            output = model(x)
        expected_shape = (2, horizon, 7)
        if tuple(output.shape) != expected_shape:
            raise AssertionError(f'{model_name}: {tuple(output.shape)} != {expected_shape}')
        shape_rows.append({
            'horizon': horizon,
            'model': model_name,
            'output_shape': str(tuple(output.shape)),
            'parameters': count_parameters(model),
        })
        del model, output

shape_df = pd.DataFrame(shape_rows)
display(shape_df)
print('全部模型前向形状检查通过。')

## 3. 运行训练

### 训练流程

每组实验都会独立完成数据加载、随机种子设置、模型初始化、训练、验证集早停、测试集预测和结果保存。主要损失为 MSE，优化器参数来自上一节的统一训练配置。

训练过程使用验证集损失选择最佳 epoch，并在连续 10 轮没有改善时提前停止。测试指标只在训练完成后计算，不参与模型选择。

### 如何启动

1. 快速检查流程时，设置 `USE_SMOKE_CONFIG=True`、`RUN_EXPERIMENTS=True`。
2. 正式实验时，保持 `USE_SMOKE_CONFIG=False`，将 `RUN_EXPERIMENTS=True`。
3. 运行中可以手动停止；重新执行本单元会根据结果文件跳过已完成组合。

运行状态会保存到 `results/run_state/ablation_rerun_seed42_ablation_state.json`，其中记录当前实验、完成数量、中断或错误信息。

In [ ]:
from scripts.run_experiments import run_one

args = SimpleNamespace(**config)
args.skip_existing = True

state_dir = ROOT / 'results' / 'run_state'
state_dir.mkdir(parents=True, exist_ok=True)
state_path = state_dir / f"{run_tag_dir}_ablation_state.json"

plan = build_plan()
pending = [row for row in plan if row['status'] == 'pending']
missing = [row for row in plan if row['status'] == 'missing_data']
completed = [row for row in plan if row['status'] == 'completed']

if RUN_ONLY_FIRST_N is not None:
    pending = pending[:RUN_ONLY_FIRST_N]

print(f'已完成: {len(completed)}，待运行: {len(pending)}，缺失数据: {len(missing)}')

def write_state(status, row=None, index=None, total=None, error=None):
    payload = {
        'status': status,
        'run_tag': config['run_tag'],
        'updated_at': time.strftime('%Y-%m-%d %H:%M:%S'),
        'completed_before_start': len(completed),
        'pending_this_run': len(pending),
        'missing_data': len(missing),
        'current_index': index,
        'current_total': total,
        'current_experiment': row,
        'error': error,
    }
    state_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')

if not RUN_EXPERIMENTS:
    print('训练安全开关关闭：本单元未启动训练。确认计划后设置 RUN_EXPERIMENTS=True。')
else:
    if missing and MISSING_DATA_POLICY == 'raise':
        raise FileNotFoundError('存在 missing_data 实验，请先补齐 data/processed。')
    if missing:
        print('MISSING_DATA_POLICY=skip，将跳过缺失数据的实验。')

    write_state('started')
    try:
        for idx, row in enumerate(pending, start=1):
            print('\n' + '=' * 100)
            print(f"[{idx}/{len(pending)}] {row['dataset']} h{row['horizon']} {row['model']}")
            write_state('running', row=row, index=idx, total=len(pending))
            run_one(args, row['dataset'], int(row['horizon']), row['model'])
            write_state('completed_one', row=row, index=idx, total=len(pending))
        write_state('finished')
        print('\n全部可运行消融实验已完成。')
    except KeyboardInterrupt:
        write_state('interrupted', error='KeyboardInterrupt')
        print('\n训练已中断。重新运行本单元会跳过已完成实验。')
    except Exception as exc:
        write_state('failed', error=repr(exc))
        raise

## 4. 汇总同批次结果

本节读取当前 run tag 下已经完成的 summary JSON，生成完整结果表。输出位置为：

- `results/ablation_csv/ablation_rerun_seed42_summary.csv`
- `results/ablation_md/ablation_rerun_seed42_summary.md`

主要指标含义：

| 指标 | 方向 | 说明 |
| --- | --- | --- |
| MSE | 越低越好 | 对较大误差惩罚更强，是本实验的主要比较指标 |
| MAE | 越低越好 | 平均绝对误差，较容易直观理解 |
| R² | 越高越好 | 衡量模型解释数据波动的程度，负值表示弱于均值预测 |
| MSE_target | 越低越好 | 只评价目标变量 OT，避免整体指标掩盖目标列表现 |
| R2_target | 越高越好 | 目标变量 OT 的决定系数 |

只有达到 24/24 时才代表正式消融矩阵完整；部分结果可以用于检查流程，但不能作为最终平均结论。

In [ ]:
from scripts.summarize_results import load_rows, format_markdown_table

rows = load_rows(
    ROOT / 'results',
    set(datasets),
    set(horizons),
    set(models),
    {config['run_tag']},
)

if not rows:
    print('当前 run_tag 尚无结果。')
else:
    result_df = pd.DataFrame(rows).sort_values(['dataset', 'horizon', 'model'])
    csv_dir = ROOT / 'results' / 'ablation_csv'
    md_dir = ROOT / 'results' / 'ablation_md'
    csv_dir.mkdir(parents=True, exist_ok=True)
    md_dir.mkdir(parents=True, exist_ok=True)
    summary_csv = csv_dir / f"{config['run_tag']}_summary.csv"
    summary_md = md_dir / f"{config['run_tag']}_summary.md"
    result_df.to_csv(summary_csv, index=False)
    summary_md.write_text(format_markdown_table(result_df), encoding='utf-8')
    print(f'Rows: {len(result_df)} / {len(plan)}')
    print(f'CSV: {summary_csv.relative_to(ROOT)}')
    print(f'MD:  {summary_md.relative_to(ROOT)}')
    display(result_df[['dataset', 'horizon', 'model', 'MSE', 'MAE', 'R2', 'MSE_target', 'R2_target', 'trained_epochs']])

## 5. 基线与消融变体公平对比

消融效果必须与同数据集、同预测步长、同批次训练的对应基线比较：

- 两个 Autoformer 变体对比 `autoformer_ablation_base`。
- 两个 PatchTST 变体对比 `patchtst_ablation_base`。

本节不读取旧 `formal_seed42`，因为旧结果的参数或训练环境可能与当前批次不同。

### 差值解释

- `delta_MSE = ablation_MSE - base_MSE`
- `delta_MSE_pct = delta_MSE / base_MSE × 100%`
- `delta_R2 = ablation_R2 - base_R2`

当 `delta_MSE > 0` 且 `delta_R2 < 0` 时，说明移除该部件后性能下降，该部件对当前任务有正贡献。反之，如果消融后 MSE 下降，不能立刻断言原部件无效，还需要结合不同数据集、预测步长、参数量、训练稳定性和实现简化程度进行讨论。

完整矩阵会形成 16 个基线—消融配对：`2 数据集 × 2 步长 × 4 消融变体`。同时生成逐任务对比表和按消融部件聚合的平均影响表。

In [ ]:
BASELINE_BY_VARIANT = {
    'autoformer_no_decomp': 'autoformer_ablation_base',
    'autoformer_no_autocorr': 'autoformer_ablation_base',
    'patchtst_no_patch': 'patchtst_ablation_base',
    'patchtst_channel_mix': 'patchtst_ablation_base',
}

all_rows = load_rows(
    ROOT / 'results',
    set(datasets),
    set(horizons),
    set(models),
    {config['run_tag']},
)

if not all_rows:
    print('请先完成至少一组基线与消融实验。')
else:
    row_index = {
        (row['dataset'], int(row['horizon']), row['model']): row
        for row in all_rows
    }
    compare_rows = []
    missing_pairs = []
    for dataset_name in datasets:
        for horizon in horizons:
            for variant, baseline_name in BASELINE_BY_VARIANT.items():
                baseline = row_index.get((dataset_name, horizon, baseline_name))
                ablation = row_index.get((dataset_name, horizon, variant))
                if baseline is None or ablation is None:
                    missing_pairs.append((dataset_name, horizon, baseline_name, variant))
                    continue
                compare_rows.append({
                    'dataset': dataset_name,
                    'horizon': horizon,
                    'base_model': baseline_name,
                    'ablation_model': variant,
                    'base_MSE': baseline['MSE'],
                    'ablation_MSE': ablation['MSE'],
                    'delta_MSE': ablation['MSE'] - baseline['MSE'],
                    'delta_MSE_pct': (ablation['MSE'] - baseline['MSE']) / (baseline['MSE'] + 1e-8) * 100,
                    'base_MAE': baseline['MAE'],
                    'ablation_MAE': ablation['MAE'],
                    'delta_MAE': ablation['MAE'] - baseline['MAE'],
                    'base_R2': baseline['R2'],
                    'ablation_R2': ablation['R2'],
                    'delta_R2': ablation['R2'] - baseline['R2'],
                    'base_MSE_target': baseline['MSE_target'],
                    'ablation_MSE_target': ablation['MSE_target'],
                    'base_params': baseline['model_params'],
                    'ablation_params': ablation['model_params'],
                    'base_train_time_seconds': baseline['train_time_seconds'],
                    'ablation_train_time_seconds': ablation['train_time_seconds'],
                })

    if missing_pairs:
        print(f'尚缺 {len(missing_pairs)} 个基线-消融配对；当前先汇总完整配对。')

    if compare_rows:
        compare_df = pd.DataFrame(compare_rows).sort_values(['dataset', 'horizon', 'base_model', 'ablation_model'])
        csv_dir = ROOT / 'results' / 'ablation_csv'
        md_dir = ROOT / 'results' / 'ablation_md'
        csv_dir.mkdir(parents=True, exist_ok=True)
        md_dir.mkdir(parents=True, exist_ok=True)
        compare_csv = csv_dir / f"{config['run_tag']}_comparison.csv"
        compare_md = md_dir / f"{config['run_tag']}_comparison.md"
        compare_df.to_csv(compare_csv, index=False)
        compare_md.write_text(format_markdown_table(compare_df), encoding='utf-8')

        effect_df = compare_df.groupby(['base_model', 'ablation_model'], as_index=False).agg(
            mean_delta_MSE=('delta_MSE', 'mean'),
            mean_delta_MSE_pct=('delta_MSE_pct', 'mean'),
            mean_delta_R2=('delta_R2', 'mean'),
        )
        effect_csv = csv_dir / f"{config['run_tag']}_mean_effect.csv"
        effect_md = md_dir / f"{config['run_tag']}_mean_effect.md"
        effect_df.to_csv(effect_csv, index=False)
        effect_md.write_text(format_markdown_table(effect_df), encoding='utf-8')

        print(f'Comparison rows: {len(compare_df)} / 16')
        print(f'CSV: {compare_csv.relative_to(ROOT)}')
        print(f'MD:  {compare_md.relative_to(ROOT)}')
        display(compare_df)
        display(effect_df)

## 6. 查看剩余任务与完成判定

最后再次扫描 24 组任务。只有状态全部为 `completed`，且公平对比表包含 16 个配对时，才可以进入可视化和报告分析。

完成后建议按以下顺序检查：

1. 确认任务进度为 24/24。
2. 确认每个 summary 中的 `model_config` 与本 notebook 参数表一致。
3. 确认对比表有 16 行，没有缺失基线或消融结果。
4. 分别讨论四个部件在 ETTh1/ETTm1、h96/h336 上的影响，而不是只看一个全局平均数。
5. 再生成消融影响图并更新实验报告，避免沿用已清理的旧消融结论。

In [ ]:
plan = build_plan()
plan_df = pd.DataFrame(plan)
display(plan_df.groupby('status').size().reset_index(name='runs'))
display(plan_df[plan_df['status'] != 'completed'])

completed_count = int((plan_df['status'] == 'completed').sum())
print(f'完成进度: {completed_count}/{len(plan_df)}')
if completed_count == len(plan_df):
    print('24 组实验均已完成，可以运行汇总与公平对比单元。')